# SSAFY Lv2 AI 실기검정 — 실전 유사 모의고사 B (연습용)

**8문항 · 100점 · 권장 120분** | 실제 공개 샘플을 참고해 구성한 **비공식 연습문제**입니다.

- 와인 시료 **13개 수치 특성 → 3개 품종(0,1,2)** 예측. 공식 샘플과 같은 데이터 종류를 사용하되 **수치 조건과 코드 작성 내용을 일부 바꿨습니다.**
- **`실행만 하세요`**는 수정하지 말고 실행, **`작성하세요`**는 `None`과 빈 부분을 완성하세요.
- 각 문항의 준비 코드는 **앞 문제 정답 없이도 실행**되도록 설계했습니다.
- 각 문제 아래 **💡 코드 힌트**를 누르면 약한 힌트만 보입니다. 정답 코드는 별도 파일에서 확인하세요.
- **인터넷·추가 데이터·별도 `practice_support.py` 불필요.** scikit-learn의 내장 Wine 데이터셋을 사용합니다.
- 정답 확인용 셀은 형태·조건 검사를 돕는 연습용 검사이며, 공식 채점기가 아닙니다.

## 데이터 / 문항 구성

- `dev`: 정답을 포함한 개발용 데이터 **142행**
- `new_samples`: 정답 없는 새 시료 **36행**
- `sample_id`: 식별자 (입력 특성 아님), `label`: 정답 품종 0·1·2
- `feature_cols`: 13개 입력 특성 열 이름

| 번호 | 주제 | 배점 |
|---|---|---:|
| 1 | 품종별 개수 및 층화 분할 | 12 |
| 2 | 품종별 평균과 산점도 | 10 |
| 3 | kNN 분류기 학습과 예측 | 12 |
| 4 | 정확도, 혼동행렬, 오분류 | 14 |
| 5 | PyTorch 신경망, 손실, 최적화 | 16 |
| 6 | 반복 학습, 손실 기록, 그래프 | 18 |
| 7 | Dropout, 평가 모드 | 10 |
| 8 | Softmax, 불확실한 시료 추출 | 8 |

## 공통 준비 — 실행만 하세요

In [ ]:
# 실행만 하세요 (이 노트북 하나만 있으면 실행됩니다)
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.linear_model import LogisticRegression

def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

wine = load_wine(as_frame=True)
all_df = wine.data.copy()
all_df.columns = [x.replace(' ', '_').replace('/', '_') for x in all_df.columns]
all_df['label'] = wine.target.to_numpy()
all_df.insert(0, 'sample_id', np.arange(1000, 1000 + len(all_df)))
feature_cols = [c for c in all_df.columns if c not in ['sample_id', 'label']]
dev, holdout = train_test_split(all_df, test_size=36, random_state=42, stratify=all_df['label'])
new_samples = holdout.drop(columns='label').copy()

def ready_train_table():
    tr, _ = train_test_split(dev, test_size=0.25, random_state=42, stratify=dev['label'])
    return tr.copy()

def ready_data():
    tr, va = train_test_split(dev, test_size=0.25, random_state=42, stratify=dev['label'])
    scaler = StandardScaler()
    X_train = scaler.fit_transform(tr[feature_cols]).astype(np.float32)
    X_valid = scaler.transform(va[feature_cols]).astype(np.float32)
    return {'X_train':X_train, 'X_valid':X_valid,
            'y_train':tr['label'].to_numpy().astype(np.int64),
            'y_valid':va['label'].to_numpy().astype(np.int64)}

def load_classification_example():
    d = ready_data()
    # 4번은 3번 답안과 관계없는 별도 예측값을 제공합니다.
    model = KNeighborsClassifier(n_neighbors=5)
    model.fit(d['X_train'], d['y_train'])
    return d['y_valid'].copy(), model.predict(d['X_valid'])

def load_reference_scores():
    d = ready_data()
    scaler = StandardScaler()
    X_dev = scaler.fit_transform(dev[feature_cols])
    X_new = scaler.transform(new_samples[feature_cols])
    clf = LogisticRegression(max_iter=500, random_state=42)
    clf.fit(X_dev, dev['label'])
    scores = clf.decision_function(X_new).astype(np.float32)
    return new_samples['sample_id'].to_numpy(), scores

seed_all(42)
print('dev:', dev.shape, '/ new_samples:', new_samples.shape)
print('입력 특성 개수:', len(feature_cols))

## 1. 품종 비율을 유지하며 데이터 나누기 | 12점

`dev`에서 각 품종의 개수를 확인한 다음, **학습 75% / 검증 25%**로 나누세요.

**(a) [4점]** 품종 번호 0, 1, 2 순서로 개수를 센 pandas Series를 `counts`에 저장하세요.

**(b) [8점]** `train_test_split`을 사용하여 `train_df`, `valid_df`에 저장하세요. `test_size=0.25`, `random_state=42`, `stratify=dev['label']`로 지정하세요. 모든 열과 기존 행 인덱스를 유지하세요.

<details><summary>💡 막힐 때만 펼쳐보기: 코드 힌트</summary>

`value_counts()`로 세고 `sort_index()`로 품종 순서를 맞춰봐. 분할할 때는 데이터프레임 전체를 넣어.

</details>

In [ ]:
# 실행만 하세요
print('dev:', dev.shape)

In [ ]:
# 작성하세요
# (a) 품종별 개수
counts = None

# (b) 학습용 / 검증용 분할
train_df = None
valid_df = None

In [ ]:
# 확인 셀 (답안을 작성한 뒤 실행)
print('counts:', counts)
print('train / valid:', getattr(train_df, 'shape', None), getattr(valid_df, 'shape', None))
if isinstance(counts, pd.Series) and isinstance(train_df, pd.DataFrame) and isinstance(valid_df, pd.DataFrame):
    assert list(counts.index) == [0,1,2]
    assert len(train_df) + len(valid_df) == len(dev)
    assert list(train_df.columns) == list(dev.columns)
    print('기본 형태 검사 통과!')

## 2. 학습 자료의 분포 확인하기 | 10점

`stats_df`는 별도 제공되는 학습 데이터입니다. `alcohol`과 `color_intensity`를 살펴보세요.

**(a) [5점]** `label`별로 두 열의 **평균**을 계산해 `group_mean`에 저장하세요. 열 순서는 `alcohol`, `color_intensity`입니다.

**(b) [5점]** 제공된 `ax`에 산점도를 그리세요. x축=`alcohol`, y축=`color_intensity`, 점의 색=`label`.

<details><summary>💡 막힐 때만 펼쳐보기: 코드 힌트</summary>

그룹별 통계는 `groupby(label)`와 `mean()`을, 산점도는 `ax.scatter(x, y, c=...)`를 떠올려봐.

</details>

In [ ]:
# 실행만 하세요
stats_df = ready_train_table()
print('통계용 시료:', len(stats_df))

In [ ]:
# 작성하세요
# (a) 품종별 평균
group_mean = None

# (b) 산점도
fig, ax = plt.subplots(figsize=(5, 3.5))
# 이곳에 산점도 코드를 작성하세요.

ax.set_xlabel('Alcohol')
ax.set_ylabel('Color intensity')
ax.set_title('Training samples by class')
plt.show()

In [ ]:
# 확인 셀
print(group_mean)
if isinstance(group_mean, pd.DataFrame):
    assert list(group_mean.columns) == ['alcohol', 'color_intensity']
    assert list(group_mean.index) == [0, 1, 2]
    print('평균 표 형태 검사 통과!')

## 3. 가까운 시료를 이용하는 분류기 학습하기 | 12점

표준화된 `X_train` (106행, 13열), `X_valid` (36행, 13열), 정답 `y_train`, `y_valid`를 제공합니다.

**(a) [6점]** `KNeighborsClassifier`로 이웃 **3개**를 사용하는 모델 `knn`을 만들고 학습하세요.

**(b) [6점]** `X_valid`를 예측해 길이 36인 `pred` 배열에 저장하세요.

<details><summary>💡 막힐 때만 펼쳐보기: 코드 힌트</summary>

분류기를 만든 뒤 학습은 `.fit(X_train, y_train)`, 검증 입력의 예측은 `.predict(...)`야.

</details>

In [ ]:
# 실행만 하세요
d = ready_data()
X_train, X_valid = d['X_train'], d['X_valid']
y_train, y_valid = d['y_train'], d['y_valid']
print(X_train.shape, X_valid.shape)

In [ ]:
# 작성하세요
# (a) 모델 생성 / 학습
knn = None

# (b) 예측
pred = None

In [ ]:
# 확인 셀
print('예측 길이:', None if pred is None else len(pred))
if pred is not None:
    assert pred.shape == (36,)
    print('기본 형태 검사 통과!')

## 4. 전체 정확도와 품종별 오류 확인하기 | 14점

별도로 제공되는 `actual`(실제)과 `guessed`(예측) 배열을 사용하세요. **3번의 `pred`는 사용하지 않습니다.**

**(a) [4점]** 정확도를 0~1 사이의 실수로 `acc`에 저장하세요.

**(b) [6점]** `confusion_matrix`에 `labels=[0,1,2]`를 지정하여 3×3 행렬 `cm`을 만드세요.

**(c) [4점]** 실제 품종 1을 다른 품종으로 잘못 예측한 개수를 `missed_1`에 저장하세요. (행=실제, 열=예측)

<details><summary>💡 막힐 때만 펼쳐보기: 코드 힌트</summary>

정확도 함수는 `accuracy_score(실제, 예측)`. 혼동행렬에서 실제 1인 행을 확인하되, 정답 1로 예측한 칸은 제외해.

</details>

In [ ]:
# 실행만 하세요
actual, guessed = load_classification_example()
print('검증 시료 수:', len(actual))

In [ ]:
# 작성하세요
acc = None
cm = None
missed_1 = None

In [ ]:
# 확인 셀
print('accuracy:', acc)
print('confusion matrix:\n', cm)
print('실제 1의 오분류:', missed_1)
if cm is not None:
    assert cm.shape == (3,3)
    print('행렬 형태 검사 통과!')

## 5. 세 품종을 구분하는 신경망 구성하기 | 16점

입력 특성 13개 → **Linear(20)** → **ReLU** → **Linear(3)** 구조의 신경망을 만드세요. 입력이 `(B,13)`이면 출력은 `(B,3)`이어야 합니다.

**(a) [8점]** `nn.Sequential`로 `net` 구성.

**(b) [4점]** `nn.CrossEntropyLoss`를 `loss_fn`에 저장.

**(c) [4점]** 학습률 **0.003**인 `torch.optim.Adam`을 `optimizer`에 저장.

**주의:** 마지막에 Softmax를 넣지 마세요. 정답은 정수 레이블입니다.

<details><summary>💡 막힐 때만 펼쳐보기: 코드 힌트</summary>

층은 `nn.Linear(입력개수, 출력개수)`로 쌓아. 옵티마이저에는 `net.parameters()`를 전달해.

</details>

In [ ]:
# 실행만 하세요
seed_all(42)
d = ready_data()
xb = torch.tensor(d['X_train'][:8], dtype=torch.float32)
yb = torch.tensor(d['y_train'][:8], dtype=torch.long)
print('입력:', tuple(xb.shape), '/ 정답:', tuple(yb.shape))

In [ ]:
# 작성하세요
net = None
loss_fn = None
optimizer = None

In [ ]:
# 확인 셀
if net is not None and loss_fn is not None and optimizer is not None:
    print('출력 형태:', tuple(net(xb).shape))
    assert net(xb).shape == (8,3)
    print('출력 형태 검사 통과!')
else:
    print('아직 작성하지 않은 변수가 있어요.')

## 6번 전용 준비 함수 안내

이 문제는 **5번의 모델과 별개**입니다. `train_one_epoch(...)`와 `validation_loss(...)`는 아래 준비 셀에 이미 완성되어 있습니다. **학습 함수 내부를 직접 작성하지 않아도 됩니다.**

## 6. 반복 학습을 실행하고 손실 기록하기 | 18점

**(a) [12점]** `for`문으로 **1~60까지 총 60 epoch** 학습하세요. 각 epoch마다 `train_one_epoch` 호출 후 `validation_loss`를 호출해 `tr_loss`, `va_loss`에 저장하고, `history`에 `{'epoch': ..., 'train_loss': ..., 'valid_loss': ...}` 딕셔너리를 추가하세요.

**(b) [6점]** `history_df`를 DataFrame으로 만들고, `ax`에 `epoch`을 x축으로 `train_loss`(label=`train`), `valid_loss`(label=`valid`) 두 선을 그리세요.

<details><summary>💡 막힐 때만 펼쳐보기: 코드 힌트</summary>

`for epoch in range(1, 61)` 형태로 생각해봐. 매번 딕셔너리를 `history.append(...)`하고, 그래프는 `ax.plot(x, y, label=...)`을 두 번 사용해.

</details>

In [ ]:
# 실행만 하세요: 6번은 앞 문제 정답이 필요 없습니다.
seed_all(42)
d6 = ready_data()
xt = torch.tensor(d6['X_train'], dtype=torch.float32)
yt = torch.tensor(d6['y_train'], dtype=torch.long)
xv = torch.tensor(d6['X_valid'], dtype=torch.float32)
yv = torch.tensor(d6['y_valid'], dtype=torch.long)
loader = DataLoader(TensorDataset(xt, yt), batch_size=16, shuffle=True)
train_net = nn.Sequential(nn.Linear(13,12), nn.ReLU(), nn.Linear(12,3))
train_loss_fn = nn.CrossEntropyLoss()
train_opt = torch.optim.Adam(train_net.parameters(), lr=0.01)

def train_one_epoch(model, loader, loss_fn, opt):
    model.train()
    losses = []
    for x_batch, y_batch in loader:
        opt.zero_grad()
        logits = model(x_batch)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        opt.step()
        losses.append(float(loss.item()))
    return float(np.mean(losses))

def validation_loss(model, x, y, loss_fn):
    model.eval()
    with torch.no_grad():
        return float(loss_fn(model(x), y).item())
print('학습 준비 완료!')

In [ ]:
# 작성하세요
# (a) 반복 학습 및 손실 기록
history = []
# 이곳에 반복문을 작성하세요.

# (b) 학습 과정 그래프
history_df = None
fig, ax = plt.subplots(figsize=(5, 3.5))
# 이곳에 두 선 그래프를 작성하세요.

ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-entropy loss')
if ax.lines:
    ax.legend()
plt.show()

In [ ]:
# 확인 셀
print('기록 개수:', len(history))
if isinstance(history_df, pd.DataFrame):
    print(history_df.head(3))
    assert len(history_df) == 60
    assert list(history_df.columns) == ['epoch','train_loss','valid_loss']
    print('기록 형태 검사 통과!')

## 7. Dropout을 적용하고 평가 모드로 예측하기 | 10점

새 신경망은 **13 → Linear(24) → ReLU → Dropout(p=0.3) → Linear(3)** 입니다.

**(a) [6점]** 위 순서대로 `nn.Sequential` 모델을 만들어 `drop_net`에 저장하세요.

**(b) [4점]** `predict_scores(model, x)` 함수에서 모델을 평가 모드로 바꾸고, 기울기 기록 없이 `x`에 대한 점수 `out`을 구해 반환하세요. 확률 변환이나 학습은 하지 않습니다.

<details><summary>💡 막힐 때만 펼쳐보기: 코드 힌트</summary>

Dropout은 ReLU **다음**, 마지막 Linear **이전**이야. 평가할 때는 `model.eval()`과 `with torch.no_grad():`를 사용해.

</details>

In [ ]:
# 실행만 하세요
seed_all(31)
sample_x = torch.tensor(ready_data()['X_valid'][:4], dtype=torch.float32)
print('sample_x:', tuple(sample_x.shape))

In [ ]:
# 작성하세요
# (a) Dropout이 있는 모델
drop_net = None

# (b) 평가용 예측 함수
def predict_scores(model, x):
    out = None
    # 이곳에 평가 모드와 예측 코드를 작성하세요.
    return out

In [ ]:
# 확인 셀
if drop_net is not None:
    out = predict_scores(drop_net, sample_x)
    print('출력 형태:', None if out is None else tuple(out.shape))
    if out is not None:
        assert out.shape == (4,3)
        assert not out.requires_grad
        print('예측 형태 검사 통과!')
else:
    print('drop_net을 먼저 완성하세요.')

## 8. 예측 확률을 이용해 재확인할 시료 찾기 | 8점

기준 모델이 새 시료 36개에 대해 계산한 `scores` 텐서 `(36,3)`과 식별자 `ids`를 제공합니다.

**(a) [4점]** 각 행에 Softmax를 적용해 `probs` `(36,3)`을 만들고, 가장 큰 확률의 품종 번호를 `labels` `(36,)`에 저장하세요. 동점이면 작은 번호를 선택하세요.

**(b) [4점]** 각 시료의 **최대 확률이 0.90 미만**인 시료의 식별자만 `ids`에서 골라 1차원 NumPy 배열 `review_ids`에 저장하세요. 정확히 0.90은 제외하세요.

<details><summary>💡 막힐 때만 펼쳐보기: 코드 힌트</summary>

`torch.softmax(scores, dim=1)`은 행별 확률, `torch.argmax(..., dim=1)`은 예측 번호야. NumPy 배열 `ids`를 인덱싱하려면 조건 마스크를 NumPy로 바꿔봐.

</details>

In [ ]:
# 실행만 하세요
ids, saved_scores = load_reference_scores()
scores = torch.tensor(saved_scores, dtype=torch.float32)
print('scores:', tuple(scores.shape), '/ ids:', len(ids))

In [ ]:
# 작성하세요
# (a) 예측 확률과 품종
probs = None
labels = None

# (b) 재확인 대상 식별자
review_ids = None

In [ ]:
# 확인 셀
print('probs:', None if probs is None else tuple(probs.shape))
print('labels:', None if labels is None else tuple(labels.shape))
print('review_ids:', review_ids)
if probs is not None and labels is not None and review_ids is not None:
    assert probs.shape == (36,3)
    assert labels.shape == (36,)
    assert isinstance(review_ids, np.ndarray) and review_ids.ndim == 1
    assert torch.allclose(probs.sum(dim=1), torch.ones(36), atol=1e-5)
    print('기본 형태 검사 통과!')

---

## 제출 전 체크

- `None`이 남아 있지 않은지, 문제에서 요구한 변수명·값·순서를 지켰는지 확인하세요.
- 각 문제의 **준비 셀 → 답안 셀 → 확인 셀** 순서로 실행하세요.
- 실제 평가에서는 **문제 안내와 감독관 지시**를 우선 따르세요. 이 파일은 개인 연습용입니다.